In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.wind_power_landing;

In [0]:
# Source file stored in Unity Catalog Volume
source_path = "/Volumes/workspace/default/wind_power_landing/wind_power_data.csv"

# Read the CSV into a Spark DataFrame
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

# Display the first records
display(df.limit(5))

In [0]:
# Count the number of records
row_count = df.count()

# Get the number of columns
column_count = len(df.columns)

print(f"Number of rows: {row_count}")
print(f"Number of columns: {column_count}")
print("Columns:", df.columns)

In [0]:
df.printSchema()

In [0]:
# Target table in Unity Catalog
bronze_table = "workspace.default.bronze_wind_power"

# Write the source data to a Delta table
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Bronze table created successfully: {bronze_table}")

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_wind_power
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM workspace.default.bronze_wind_power;

In [0]:
from pyspark.sql import functions as F

# Read the Bronze Delta table
bronze_df = spark.table("workspace.default.bronze_wind_power")

# Dataset dimensions
print(f"Rows: {bronze_df.count()}")
print(f"Columns: {len(bronze_df.columns)}")

# Check duplicate production IDs
duplicate_ids = (
    bronze_df
    .groupBy("production_id")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate production IDs: {duplicate_ids.count()}")

# Check null values in every column
null_counts = bronze_df.select([
    F.count(
        F.when(F.col(column).isNull(), 1)
    ).alias(column)
    for column in bronze_df.columns
])

display(null_counts)

In [0]:
status_distribution = (
    bronze_df
    .groupBy("status")
    .count()
    .orderBy(F.desc("count"))
)

display(status_distribution)

In [0]:
# Online turbines producing zero energy
online_zero_energy = bronze_df.filter(
    (F.col("status") == "Online")
    & (F.col("energy_produced") == 0)
).count()

# Non-online turbines producing positive energy
non_online_positive_energy = bronze_df.filter(
    (F.col("status") != "Online")
    & (F.col("energy_produced") > 0)
).count()

print(f"Online with zero energy: {online_zero_energy}")
print(f"Non-online with positive energy: {non_online_positive_energy}")

In [0]:
from pyspark.sql import functions as F

# Read the Bronze Delta table
bronze_df = spark.table("workspace.default.bronze_wind_power")

# Build a timestamp from the date and time columns
silver_df = bronze_df.withColumn(
    "measurement_timestamp",
    F.to_timestamp(
        F.concat_ws(" ", F.col("date"), F.col("time")),
        "yyyy-MM-dd HH-mm-ss"
    )
)

display(
    silver_df.select(
        "date",
        "time",
        "measurement_timestamp",
        "turbine_name",
        "energy_produced"
    ).limit(10)
)

In [0]:
from pyspark.sql import functions as F

# Count rows where the timestamp could not be created
invalid_timestamps = silver_df.filter(
    F.col("measurement_timestamp").isNull()
).count()

# Count null values in each column
null_counts = silver_df.select([
    F.count(
        F.when(F.col(column).isNull(), 1)
    ).alias(column)
    for column in silver_df.columns
])

print(f"Rows with invalid timestamps: {invalid_timestamps}")
display(null_counts)

In [0]:
# Check duplicate production IDs
duplicate_ids = (
    silver_df
    .groupBy("production_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check negative wind speeds
negative_wind_speeds = silver_df.filter(
    F.col("wind_speed") < 0
).count()

# Check negative energy production
negative_energy = silver_df.filter(
    F.col("energy_produced") < 0
).count()

# Check missing turbine names
missing_turbines = silver_df.filter(
    F.col("turbine_name").isNull()
).count()

print(f"Duplicate production IDs: {duplicate_ids}")
print(f"Negative wind speeds: {negative_wind_speeds}")
print(f"Negative energy values: {negative_energy}")
print(f"Missing turbine names: {missing_turbines}")

In [0]:
# Count duplicate measurements for the same turbine and timestamp
duplicate_measurements = (
    silver_df
    .groupBy("turbine_name", "measurement_timestamp")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_measurements_count = duplicate_measurements.count()

print(f"Duplicate turbine measurements: {duplicate_measurements_count}")